# 02 — Cleaning & Merging

Reads all raw INMET files, applies cleaning transformations, and produces a single
merged DataFrame saved to `data/processed/`.

Cleaning steps:
- Rename columns to snake_case
- Merge `Data` + `Hora UTC` into a single `timestamp` column
- Convert `latitude` and `longitude` from string to float
- Normalize station names (B807 → B825 for Porto Alegre Belém Novo)
- Flag and document stations with high null rates

In [2]:
import re
from io import StringIO
import pandas as pd
from pathlib import Path

ROOT = Path().resolve()
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"

print(ROOT)
print(RAW)
print(PROCESSED)

D:\projects\air_quality_brazil
D:\projects\air_quality_brazil\data\raw
D:\projects\air_quality_brazil\data\processed


In [3]:
def read_inmet_file(filepath):
    """
    Reads a single INMET automatic station CSV file and returns a clean DataFrame.

    INMET files have 8 rows of station metadata before the actual column headers,
    use semicolon as separator, comma as decimal, and latin-1 encoding.
    Some values are recorded as bare comma-decimals (e.g. ',4' instead of '0,4')
    which we fix before passing to pandas.
    """

    with open(filepath, encoding="latin-1") as f:
        lines = f.readlines()

    # Extract and parse the 8 metadata rows
    meta_lines = [line.strip() for line in lines[:8]]
    meta = {}
    meta_keys = ["regiao", "uf", "estacao", "codigo_wmo", "latitude", "longitude", "altitude", "data_fundacao"]
    for key, line in zip(meta_keys, meta_lines):
        meta[key] = line.split(";")[1].strip() if ";" in line else ""

    # Fix bare comma-decimals in data rows (lines 9 onward, skipping header on line 8)
    # A bare comma-decimal looks like ';,4;' — a semicolon followed immediately by a comma
    # We insert a '0' before the comma to make it ';0,4;' which pandas can parse correctly
    fixed_lines = lines[:9]  # keep metadata + header row untouched
    for line in lines[9:]:
        # Replace any occurrence of ';,' with ';0,' to fix bare decimals
        line = re.sub(r';,', ';0,', line)
        fixed_lines.append(line)

    # Join fixed lines into a single string and pass to pandas via StringIO
    # This avoids writing a temporary file to disk
    content = "".join(fixed_lines)

    df = pd.read_csv(
        StringIO(content),
        sep=";",
        skiprows=8,
        decimal=",",
        encoding="latin-1",
        low_memory=False,
    )

    # Drop the trailing empty column from trailing semicolons
    df = df.drop(columns=df.columns[-1])

    # Attach station metadata
    df["estacao"] = meta["estacao"]
    df["codigo_wmo"] = meta["codigo_wmo"]
    df["uf"] = meta["uf"]
    df["latitude"] = meta["latitude"]
    df["longitude"] = meta["longitude"]

    return df, meta

In [4]:
# Load all CSV files and concatenate into one DataFrame
# This is the foundation everything else in this notebook builds on

all_files = sorted((RAW / "INMET").glob("**/*.CSV"))

dfs = []  # collect each file's DataFrame here before concatenating

for filepath in all_files:
    try:
        df, meta = read_inmet_file(filepath)
        # Add the year as a column — useful for filtering and grouping later
        df["year"] = int(filepath.parent.name)
        dfs.append(df)
    except Exception as e:
        print(f"ERROR in {filepath.name}: {e}")

# Concatenate all DataFrames into one
# ignore_index resets the index so rows are numbered 0, 1, 2... continuously
# instead of each file's index restarting at 0
df_raw = pd.concat(dfs, ignore_index=True)

print(f"Total rows: {len(df_raw):,}")
print(f"Total columns: {len(df_raw.columns)}")
print(f"Years covered: {sorted(df_raw['year'].unique())}")
print(f"Stations: {sorted(df_raw['estacao'].unique())}")

Total rows: 272,472
Total columns: 25
Years covered: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Stations: ['BELO HORIZONTE (PAMPULHA)', 'BELO HORIZONTE - CERCADINHO', 'BELO HORIZONTE - PAMPULHA', 'BELO HORIZONTE - SANTO AGOSTINHO', 'BRASILIA', 'MANAUS', 'PORTO ALEGRE - BELEM NOVO', 'PORTO ALEGRE - JARDIM BOTANICO', 'PORTO ALEGRE- BELEM NOVO', 'SAO PAULO - INTERLAGOS', 'SAO PAULO - MIRANTE']


In [5]:
# Verify row counts per station per year
# Expected: 8760 rows for normal years, 8784 for 2024 (leap year)
# Deviations tell us about data loss from the comma-decimal fix

counts = (
    df_raw.groupby(["estacao", "year"])
    .size()
    .reset_index(name="rows")
)

print(counts.to_string(index=False))

                         estacao  year  rows
       BELO HORIZONTE (PAMPULHA)  2022  8760
       BELO HORIZONTE (PAMPULHA)  2023  8760
       BELO HORIZONTE (PAMPULHA)  2024  8784
     BELO HORIZONTE - CERCADINHO  2022  8760
     BELO HORIZONTE - CERCADINHO  2023  8760
     BELO HORIZONTE - CERCADINHO  2024  8784
     BELO HORIZONTE - CERCADINHO  2025  8760
       BELO HORIZONTE - PAMPULHA  2025  8760
BELO HORIZONTE - SANTO AGOSTINHO  2025  3024
                        BRASILIA  2022  8760
                        BRASILIA  2023  8760
                        BRASILIA  2024  8784
                        BRASILIA  2025  8760
                          MANAUS  2022  8760
                          MANAUS  2023  8760
                          MANAUS  2024  8784
                          MANAUS  2025  8760
       PORTO ALEGRE - BELEM NOVO  2025  5880
  PORTO ALEGRE - JARDIM BOTANICO  2022  8760
  PORTO ALEGRE - JARDIM BOTANICO  2023  8760
  PORTO ALEGRE - JARDIM BOTANICO  2024  8784
  PORTO AL

In [6]:
# Normalize station names — same physical station, different spellings across years
# We standardize to a single consistent name per station

station_name_map = {
    "BELO HORIZONTE (PAMPULHA)": "BELO HORIZONTE - PAMPULHA",  # 2022-2024 vs 2025
    "PORTO ALEGRE- BELEM NOVO": "PORTO ALEGRE - BELEM NOVO",   # missing space before hyphen
}

df_raw["estacao"] = df_raw["estacao"].replace(station_name_map)

# Also normalize the station code for Porto Alegre Belem Novo
# B807 (2022-2024) was renamed to B825 (2025) — same physical station
df_raw["codigo_wmo"] = df_raw["codigo_wmo"].replace({"B807": "B825"})

# Verify — should now show clean unique names
print(df_raw["estacao"].unique())
print(df_raw["codigo_wmo"].unique())

<ArrowStringArray>
[                        'BRASILIA',                           'MANAUS',
   'PORTO ALEGRE - JARDIM BOTANICO',        'PORTO ALEGRE - BELEM NOVO',
        'BELO HORIZONTE - PAMPULHA',      'BELO HORIZONTE - CERCADINHO',
              'SAO PAULO - MIRANTE',           'SAO PAULO - INTERLAGOS',
 'BELO HORIZONTE - SANTO AGOSTINHO']
Length: 9, dtype: str
<ArrowStringArray>
['A001', 'A101', 'A801', 'B825', 'A521', 'F501', 'A701', 'A771', 'A572']
Length: 9, dtype: str


In [7]:
# Rename columns from long Portuguese names to short snake_case
# This makes the DataFrame much easier to work with throughout the project

column_map = {
    "Data": "date",
    "Hora UTC": "time_utc",
    "PRECIPITAÇÃO TOTAL, HORÁRIO (mm)": "precip_mm",
    "PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)": "pressure_mb",
    "PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)": "pressure_max_mb",
    "PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)": "pressure_min_mb",
    "RADIACAO GLOBAL (Kj/m²)": "radiation_kjm2",
    "TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)": "temp_c",
    "TEMPERATURA DO PONTO DE ORVALHO (°C)": "dewpoint_c",
    "TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)": "temp_max_c",
    "TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)": "temp_min_c",
    "TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)": "dewpoint_max_c",
    "TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)": "dewpoint_min_c",
    "UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)": "humidity_max_pct",
    "UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)": "humidity_min_pct",
    "UMIDADE RELATIVA DO AR, HORARIA (%)": "humidity_pct",
    "VENTO, DIREÇÃO HORARIA (gr) (° (gr))": "wind_dir_deg",
    "VENTO, RAJADA MAXIMA (m/s)": "wind_gust_ms",
    "VENTO, VELOCIDADE HORARIA (m/s)": "wind_speed_ms",
}

df_raw = df_raw.rename(columns=column_map)

# Verify the rename worked
print(df_raw.columns.tolist())

['date', 'time_utc', 'precip_mm', 'pressure_mb', 'pressure_max_mb', 'pressure_min_mb', 'radiation_kjm2', 'temp_c', 'dewpoint_c', 'temp_max_c', 'temp_min_c', 'dewpoint_max_c', 'dewpoint_min_c', 'humidity_max_pct', 'humidity_min_pct', 'humidity_pct', 'wind_dir_deg', 'wind_gust_ms', 'wind_speed_ms', 'estacao', 'codigo_wmo', 'uf', 'latitude', 'longitude', 'year']


In [8]:
# Combine date and time into a single timestamp column
# Current format: date='2022/01/01', time_utc='0000 UTC'
# We strip ' UTC' from the time, then parse the combined string

df_raw["timestamp"] = pd.to_datetime(
    df_raw["date"] + " " + df_raw["time_utc"].str.replace(" UTC", "", regex=False),
    format="%Y/%m/%d %H%M",
    utc=True,  # mark as UTC since that's what INMET records
)

# Drop the now-redundant separate columns
df_raw = df_raw.drop(columns=["date", "time_utc"])

# Move timestamp to the front for readability
cols = ["timestamp"] + [c for c in df_raw.columns if c != "timestamp"]
df_raw = df_raw[cols]

# Verify
print(df_raw["timestamp"].dtype)
print(df_raw["timestamp"].head(3))
print(f"\nEarliest: {df_raw['timestamp'].min()}")
print(f"Latest:   {df_raw['timestamp'].max()}")

datetime64[us, UTC]
0   2022-01-01 00:00:00+00:00
1   2022-01-01 01:00:00+00:00
2   2022-01-01 02:00:00+00:00
Name: timestamp, dtype: datetime64[us, UTC]

Earliest: 2022-01-01 00:00:00+00:00
Latest:   2025-12-31 23:00:00+00:00


In [9]:
# Convert latitude and longitude from string to float
# They were read as strings because of the comma decimal separator
# e.g. '-23,49638888' -> -23.49638888

df_raw["latitude"] = df_raw["latitude"].str.replace(",", ".").astype(float)
df_raw["longitude"] = df_raw["longitude"].str.replace(",", ".").astype(float)

# Verify
print(df_raw[["estacao", "latitude", "longitude"]].drop_duplicates().to_string(index=False))

                         estacao   latitude  longitude
                        BRASILIA -15.789444 -47.925833
                          MANAUS  -3.103333 -60.016389
  PORTO ALEGRE - JARDIM BOTANICO -30.053611 -51.174722
       PORTO ALEGRE - BELEM NOVO -30.186111 -51.178056
       BELO HORIZONTE - PAMPULHA -19.883889 -43.969444
     BELO HORIZONTE - CERCADINHO -19.980000 -43.958611
             SAO PAULO - MIRANTE -23.496389 -46.620000
          SAO PAULO - INTERLAGOS -23.724501 -46.677501
          SAO PAULO - INTERLAGOS -23.724444 -46.677500
       PORTO ALEGRE - BELEM NOVO -30.186111 -51.178333
BELO HORIZONTE - SANTO AGOSTINHO -19.933889 -43.952222
             SAO PAULO - MIRANTE -23.496289 -46.620067


In [10]:
# Some stations have slightly different coordinates across years
# due to GPS recalibration in INMET metadata — differences are <100m
# We standardize by taking the most frequent coordinate per station

for col in ["latitude", "longitude"]:
    # For each row, replace coordinate with the mode (most common value)
    # for that station
    mode_coords = df_raw.groupby("estacao")[col].agg(
        lambda x: x.mode()[0]  # mode() returns a Series, take first value
    )
    df_raw[col] = df_raw["estacao"].map(mode_coords)

# Verify — should now show exactly one row per station
print(df_raw[["estacao", "latitude", "longitude"]].drop_duplicates().to_string(index=False))

                         estacao   latitude  longitude
                        BRASILIA -15.789444 -47.925833
                          MANAUS  -3.103333 -60.016389
  PORTO ALEGRE - JARDIM BOTANICO -30.053611 -51.174722
       PORTO ALEGRE - BELEM NOVO -30.186111 -51.178056
       BELO HORIZONTE - PAMPULHA -19.883889 -43.969444
     BELO HORIZONTE - CERCADINHO -19.980000 -43.958611
             SAO PAULO - MIRANTE -23.496389 -46.620000
          SAO PAULO - INTERLAGOS -23.724444 -46.677500
BELO HORIZONTE - SANTO AGOSTINHO -19.933889 -43.952222


In [11]:
# Summary of the cleaned DataFrame so far
print(f"Shape: {df_raw.shape}")
print(f"\nDtypes:")
print(df_raw.dtypes)
print(f"\nNull counts per column:")
print(df_raw.isnull().sum().to_string())

Shape: (272472, 24)

Dtypes:
timestamp           datetime64[us, UTC]
precip_mm                       float64
pressure_mb                     float64
pressure_max_mb                 float64
pressure_min_mb                 float64
radiation_kjm2                  float64
temp_c                          float64
dewpoint_c                      float64
temp_max_c                      float64
temp_min_c                      float64
dewpoint_max_c                  float64
dewpoint_min_c                  float64
humidity_max_pct                float64
humidity_min_pct                float64
humidity_pct                    float64
wind_dir_deg                    float64
wind_gust_ms                    float64
wind_speed_ms                   float64
estacao                             str
codigo_wmo                          str
uf                                  str
latitude                        float64
longitude                       float64
year                              int64
dtype: obje

In [12]:
# Save the cleaned DataFrame to processed/
# We use parquet instead of CSV for two reasons:
# 1. It preserves dtypes (timestamps, floats) — no re-parsing needed when loading
# 2. It's significantly smaller and faster to read than CSV

output_path = PROCESSED / "inmet_clean.parquet"
df_raw.to_parquet(output_path, index=False)

print(f"Saved to {output_path}")
print(f"File size: {output_path.stat().st_size / 1024 / 1024:.2f} MB")

Saved to D:\projects\air_quality_brazil\data\processed\inmet_clean.parquet
File size: 5.39 MB


In [13]:
# Reload from parquet and verify everything survived the round-trip
df_check = pd.read_parquet(PROCESSED / "inmet_clean.parquet")

print(f"Shape: {df_check.shape}")
print(f"\nDtypes:")
print(df_check.dtypes)
print(f"\nFirst row:")
print(df_check.iloc[0])

Shape: (272472, 24)

Dtypes:
timestamp           datetime64[us, UTC]
precip_mm                       float64
pressure_mb                     float64
pressure_max_mb                 float64
pressure_min_mb                 float64
radiation_kjm2                  float64
temp_c                          float64
dewpoint_c                      float64
temp_max_c                      float64
temp_min_c                      float64
dewpoint_max_c                  float64
dewpoint_min_c                  float64
humidity_max_pct                float64
humidity_min_pct                float64
humidity_pct                    float64
wind_dir_deg                    float64
wind_gust_ms                    float64
wind_speed_ms                   float64
estacao                             str
codigo_wmo                          str
uf                                  str
latitude                        float64
longitude                       float64
year                              int64
dtype: obje

In [14]:
# Null analysis per station — we need to understand which stations
# have problematic null rates before deciding how to handle them
# We focus on temp_c as our proxy: it's the most important variable
# and a good indicator of overall station health

null_summary = (
    df_raw.groupby(["estacao", "year"])
    .agg(
        total_rows=("temp_c", "count"),        # non-null count
        null_temp=("temp_c", lambda x: x.isnull().sum()),
        total=("temp_c", "size"),
    )
    .assign(null_pct=lambda x: (x["null_temp"] / x["total"] * 100).round(1))
    .reset_index()
)

# Only show rows with any nulls — clean stations are not interesting here
print(null_summary[null_summary["null_pct"] > 0].to_string(index=False))

                         estacao  year  total_rows  null_temp  total  null_pct
     BELO HORIZONTE - CERCADINHO  2022        8678         82   8760       0.9
     BELO HORIZONTE - CERCADINHO  2025        8727         33   8760       0.4
       BELO HORIZONTE - PAMPULHA  2022        8752          8   8760       0.1
       BELO HORIZONTE - PAMPULHA  2025        8748         12   8760       0.1
BELO HORIZONTE - SANTO AGOSTINHO  2025        2896        128   3024       4.2
                        BRASILIA  2023        8755          5   8760       0.1
                        BRASILIA  2024        8758         26   8784       0.3
                        BRASILIA  2025        8739         21   8760       0.2
                          MANAUS  2022        8695         65   8760       0.7
                          MANAUS  2023        8636        124   8760       1.4
                          MANAUS  2024        8700         84   8784       1.0
                          MANAUS  2025        8687  

In [ ]:
# Check the distribution of valid readings for Belem Novo 2025
# If valid rows are spread across the year, the station has value
# If they're clustered, it's effectively a very short partial year

belem_2025 = df_raw[
    (df_raw["estacao"] == "PORTO ALEGRE - BELEM NOVO") &
    (df_raw["year"] == 2025)
].copy()

# Count valid temp readings per month
belem_2025["month"] = belem_2025["timestamp"].dt.month
monthly = belem_2025.groupby("month").agg(
    total_hours=("temp_c", "size"),
    valid_hours=("temp_c", "count"),
).assign(valid_pct=lambda x: (x["valid_hours"] / x["total_hours"] * 100).round(1))

print(monthly.to_string())

       total_hours  valid_hours  valid_pct
month                                     
5              744            0        0.0
6              720            0        0.0
7              744            0        0.0
8              744            0        0.0
9              720            0        0.0
10             744          437       58.7
11             720          720      100.0
12             744          742       99.7


In [ ]:
# Exclude Porto Alegre Belem Novo 2025 — 67.7% null, unusable
# Considered keeping partial data but determined the coverage gap made the contribution
# unreliable
before = len(df_raw)

df_raw = df_raw[
    ~((df_raw["estacao"] == "PORTO ALEGRE - BELEM NOVO") & (df_raw["year"] == 2025))
]

after = len(df_raw)
print(f"Removed {before - after:,} rows — Porto Alegre Belém Novo 2025")
print(f"Remaining rows: {after:,}")

Removed 5,880 rows — Porto Alegre Belém Novo 2025
Remaining rows: 266,592


In [ ]:
# Add a city column mapping each station to its city
# This is more useful for aggregation and dashboard grouping than the full station name
# which includes neighborhood/district

city_map = {
    "BRASILIA": "Brasília",
    "MANAUS": "Manaus",
    "PORTO ALEGRE - JARDIM BOTANICO": "Porto Alegre",
    "PORTO ALEGRE - BELEM NOVO": "Porto Alegre",
    "BELO HORIZONTE - PAMPULHA": "Belo Horizonte",
    "BELO HORIZONTE - CERCADINHO": "Belo Horizonte",
    "BELO HORIZONTE - SANTO AGOSTINHO": "Belo Horizonte",
    "SAO PAULO - MIRANTE": "São Paulo",
    "SAO PAULO - INTERLAGOS": "São Paulo",
}

df_raw["city"] = df_raw["estacao"].map(city_map)

# Verify — no nulls means every station was mapped correctly
print(df_raw["city"].value_counts())
print(f"\nUnmapped stations: {df_raw['city'].isnull().sum()}")

city
Belo Horizonte    73152
São Paulo         70128
Porto Alegre      53184
Brasília          35064
Manaus            35064
Name: count, dtype: int64

Unmapped stations: 0


In [18]:
# Interpolate short null gaps per station
# Sort by station + timestamp first to ensure interpolation runs along the time axis
# correctly

# Columns to interpolate — all numeric weather measurements
# Exclude metadata columns and anything categorical
weather_cols = [
    "precip_mm", "pressure_mb", "pressure_max_mb", "pressure_min_mb",
    "radiation_kjm2", "temp_c", "dewpoint_c", "temp_max_c", "temp_min_c",
    "dewpoint_max_c", "dewpoint_min_c", "humidity_max_pct", "humidity_min_pct",
    "humidity_pct", "wind_dir_deg", "wind_gust_ms", "wind_speed_ms",
]

# Sort so interpolation runs along the time axis within each station
df_raw = df_raw.sort_values(["estacao", "timestamp"]).reset_index(drop=True)

# Interpolate per station group, limit to 2 consecutive nulls
df_raw[weather_cols] = (
    df_raw.groupby("estacao")[weather_cols]
    .transform(lambda x: x.interpolate(method="linear", limit=2))
)

# Verify null reduction — compare to previous null counts
print("Null counts after interpolation:")
print(df_raw[weather_cols].isnull().sum().to_string())

Null counts after interpolation:
precip_mm            1593
pressure_mb           620
pressure_max_mb       636
pressure_min_mb       636
radiation_kjm2      65097
temp_c               1594
dewpoint_c           1594
temp_max_c           1610
temp_min_c           1610
dewpoint_max_c       1610
dewpoint_min_c       1610
humidity_max_pct     1610
humidity_min_pct     1610
humidity_pct         1594
wind_dir_deg         3900
wind_gust_ms         3983
wind_speed_ms        3915


In [19]:
# Save final cleaned DataFrame to processed/
# This overwrites the earlier save which didn't have null handling, city column, or the
# Belem Novo 2025 exclusion

output_path = PROCESSED / "inmet_clean.parquet"
df_raw.to_parquet(output_path, index=False)

print(f"Saved to {output_path}")
print(f"Shape: {df_raw.shape}")
print(f"File size: {output_path.stat().st_size / 1024 / 1024:.2f} MB")

Saved to D:\projects\air_quality_brazil\data\processed\inmet_clean.parquet
Shape: (266592, 25)
File size: 5.74 MB


# =============================================================
# CLEANING SUMMARY — decisions made in this notebook
# =============================================================

INPUT
- 33 raw INMET CSV files across 2022–2025
- 9 automatic stations across 5 cities

STRUCTURAL FIXES
- Bare comma-decimals (e.g. ',4') fixed via regex before parsing
- Column names renamed to snake_case
- date + time_utc merged into single UTC timestamp
- latitude/longitude converted from string to float
- Coordinate inconsistencies across years standardized to mode per station
- Station names normalized (PAMPULHA spelling, BELEM NOVO spacing)
- Station code B807 → B825 normalized (same physical station, renamed 2025)
- City column added mapping stations to 5 cities

EXCLUSIONS
- Porto Alegre Belém Novo 2025: 67.7% null temperature readings
  Valid data only in Nov–Dec 2025; Jardim Botânico covers city fully
  Removed 5,880 rows

NULL HANDLING
- Short gaps (≤2 consecutive hours): linear interpolation
- Long gaps (>2 hours): left as NaN — not fabricated
- radiation_kjm2 retains high null count (~65k) — legitimate nighttime nulls

OUTPUT
- 266,592 rows × 25 columns
- Saved to data/processed/inmet_clean.parquet
- All dtypes preserved: timestamps as datetime64[us, UTC], numerics as float64

STATIONS FLAGGED FOR ANALYSIS
- SAO PAULO - INTERLAGOS 2024: 11.2% null temperature (sensor degradation)
- BELO HORIZONTE - SANTO AGOSTINHO 2025: partial year from Aug 28, 4.2% null
- PORTO ALEGRE - BELEM NOVO 2022: partial year from Dec 8, 3.5% null